In [5]:
# shortjokes_lightgcn_demo.ipynb
# Demo: LightGCN on your ShortJokes (ABCD) ratings

import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path

# -----------------------------
# PART 0 — Reproducibility setup
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("\n==============================")
print("PART 0 — Reproducibility setup")
print("==============================")
print(f"Seed set to: {SEED}")
print(f"PyTorch version: {torch.__version__}")

# -----------------------------
# PART 1 — Set file paths
# -----------------------------
print("\n===================")
print("PART 1 — File paths")
print("===================")

BASE = Path(r"C:\Users\timil\Downloads\Joke_Project_Diss2026")
ratings_path = BASE / "shortjokes_ratings_combined.csv"

print("Base folder:", BASE)
print("Ratings file:", ratings_path)
print("Base folder exists?:", BASE.exists())
print("Ratings file exists?:", ratings_path.exists())

# -----------------------------
# PART 2 — Load & inspect dataset
# -----------------------------
print("\n================================")
print("PART 2 — Load & inspect dataset")
print("================================")

df = pd.read_csv(ratings_path)

# Basic cleanup
df["user_id"] = df["user_id"].astype(str).str.strip()
df["joke_id"] = df["joke_id"].astype(str).str.strip()

has_text = "joke_text" in df.columns

print("Loaded:", ratings_path)
print("Columns:", list(df.columns))
print("Has joke_text?:", has_text)

print("\nFirst 5 rows:")
display(df.head())  # If you're in Jupyter, display looks nicer than print

print("\nDataset summary:")
print("Rows:", len(df), "| Users:", df["user_id"].nunique(), "| Jokes:", df["joke_id"].nunique())

print("\nRating distribution:")
print(df["rating"].value_counts(dropna=False))



PART 0 — Reproducibility setup
Seed set to: 42
PyTorch version: 2.9.1+cpu

PART 1 — File paths
Base folder: C:\Users\timil\Downloads\Joke_Project_Diss2026
Ratings file: C:\Users\timil\Downloads\Joke_Project_Diss2026\shortjokes_ratings_combined.csv
Base folder exists?: True
Ratings file exists?: True

PART 2 — Load & inspect dataset
Loaded: C:\Users\timil\Downloads\Joke_Project_Diss2026\shortjokes_ratings_combined.csv
Columns: ['user_id', 'joke_id', 'joke_text', 'rating', 'source_file']
Has joke_text?: True

First 5 rows:


,user_id,joke_id,joke_text,rating,source_file
0,A,15270,Where does beef come from? Cowschwitz.,1,ShortJokes-Userdata-edges-A.csv
1,A,28989,What do Angels fans and gay men both have in c...,0,ShortJokes-Userdata-edges-A.csv
2,A,40594,What's the difference between a garbanzo bean ...,-1,ShortJokes-Userdata-edges-A.csv
3,A,41814,Trying to take the best instagram picture ever...,1,ShortJokes-Userdata-edges-A.csv
4,A,46109,Chess makes us to realize our life!!! Chess sa...,1,ShortJokes-Userdata-edges-A.csv



Dataset summary:
Rows: 196 | Users: 4 | Jokes: 49

Rating distribution:
rating
 1    86
 0    62
-1    48
Name: count, dtype: int64


In [6]:
# -----------------------------
# PART 3 — ID mapping (user/item -> indices)
# -----------------------------
print("\n=======================================")
print("PART 3 — ID mapping (user/item indices)")
print("=======================================")

# Create index mappings
user_ids = sorted(df["user_id"].unique())
item_ids = sorted(df["joke_id"].unique())

user2idx = {u: i for i, u in enumerate(user_ids)}
item2idx = {it: i for i, it in enumerate(item_ids)}

idx2user = {i: u for u, i in user2idx.items()}
idx2item = {i: it for it, i in item2idx.items()}

# Add indices to dataframe
df["user_idx"] = df["user_id"].map(user2idx)
df["item_idx"] = df["joke_id"].map(item2idx)

n_users = len(user2idx)
n_items = len(item2idx)

print("Users (user_id -> user_idx):", user2idx)
print("Number of users:", n_users)
print("Number of jokes/items:", n_items)

print("\nSample rows with indices:")
display(df[["user_id", "user_idx", "joke_id", "item_idx", "rating"]].head())

# -----------------------------
# PART 4 — Build implicit positives for LightGCN
# -----------------------------
print("\n============================================")
print("PART 4 — Implicit positives (rating == 1)")
print("============================================")

# LightGCN typically uses implicit feedback (positives only)
pos_df = df[df["rating"] == 1].copy()

print("Positive edges (rating==1):", len(pos_df))

print("\nPositives per user:")
pos_per_user = pos_df.groupby("user_id")["item_idx"].count().sort_values(ascending=False)
print(pos_per_user)

print("\nPositives per user (as % of their ratings):")
user_total = df.groupby("user_id")["rating"].count()
pos_pct = (pos_per_user / user_total * 100).round(1)
print(pos_pct.fillna(0).astype(str) + "%")



PART 3 — ID mapping (user/item indices)
Users (user_id -> user_idx): {'A': 0, 'B': 1, 'C': 2, 'D': 3}
Number of users: 4
Number of jokes/items: 49

Sample rows with indices:


,user_id,user_idx,joke_id,item_idx,rating
0,A,0,15270,13,1
1,A,0,28989,30,0
2,A,0,40594,31,-1
3,A,0,41814,32,1
4,A,0,46109,33,1



PART 4 — Implicit positives (rating == 1)
Positive edges (rating==1): 86

Positives per user:
user_id
A    28
C    25
D    17
B    16
Name: item_idx, dtype: int64

Positives per user (as % of their ratings):
user_id
A    57.1%
B    32.7%
C    51.0%
D    34.7%
dtype: object


In [12]:
# -----------------------------
# PART 5 — Train/Val/Test split (per-user holdout, no leakage)
# -----------------------------
print("\n====================================================")
print("PART 5 — Train/Val/Test split (per-user holdout)")
print("====================================================")

import random

# Strategy:
# If a user has >= 3 positive jokes:
#   train = all but last 2, val = 1, test = 1
# If a user has 2 positives:
#   train = 1, test = 1 (no val)
# If a user has 1 positive:
#   train = 1 (can't evaluate, but keep pipeline consistent)

train_edges, val_edges, test_edges = [], [], []

for u, group in pos_df.groupby("user_idx"):
    items = group["item_idx"].tolist()
    random.shuffle(items)

    if len(items) >= 3:
        train_items = items[:-2]
        val_item = items[-2]
        test_item = items[-1]

        train_edges += [(u, i) for i in train_items]
        val_edges += [(u, val_item)]
        test_edges += [(u, test_item)]

    elif len(items) == 2:
        train_edges += [(u, items[0])]
        test_edges += [(u, items[1])]

    elif len(items) == 1:
        train_edges += [(u, items[0])]

print("Split sizes:")
print(" - Train edges:", len(train_edges))
print(" - Val edges:  ", len(val_edges))
print(" - Test edges: ", len(test_edges))

# Build lookup sets for fast masking and negative sampling later
train_user_pos = {u: set() for u in range(n_users)}
val_user_pos   = {u: set() for u in range(n_users)}
test_user_pos  = {u: set() for u in range(n_users)}

for u, i in train_edges:
    train_user_pos[u].add(i)
for u, i in val_edges:
    val_user_pos[u].add(i)
for u, i in test_edges:
    test_user_pos[u].add(i)

print("\nPer-user split summary (positives):")
for u in range(n_users):
    uid = idx2user[u]
    print(f"User {uid}: train={len(train_user_pos[u])}, val={len(val_user_pos[u])}, test={len(test_user_pos[u])}")





PART 5 — Train/Val/Test split (per-user holdout)
Split sizes:
 - Train edges: 78
 - Val edges:   4
 - Test edges:  4

Per-user split summary (positives):
User A: train=26, val=1, test=1
User B: train=14, val=1, test=1
User C: train=23, val=1, test=1
User D: train=15, val=1, test=1


In [8]:
# -----------------------------
# PART 6 — Build normalized adjacency matrix (LightGCN graph)
# -----------------------------
print("\n===================================================")
print("PART 6 — Build normalized adjacency (user–item graph)")
print("===================================================")

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

def build_norm_adj(train_edges, n_users, n_items, device="cpu"):
    """
    Builds a normalized sparse adjacency matrix for a bipartite graph.

    Nodes are arranged as:
      [0 .. n_users-1] = users
      [n_users .. n_users+n_items-1] = items

    For each (u, i) edge, we add:
      u <-> (n_users + i)

    LightGCN typically uses symmetric normalization:
      A_hat[u,v] = 1 / sqrt(deg[u] * deg[v])
    """
    # Create undirected edges
    rows, cols = [], []
    for u, i in train_edges:
        v = n_users + i
        rows += [u, v]
        cols += [v, u]

    rows = torch.tensor(rows, dtype=torch.long, device=device)
    cols = torch.tensor(cols, dtype=torch.long, device=device)

    N = n_users + n_items

    # Degree calculation
    deg = torch.zeros(N, device=device)
    deg.index_add_(0, rows, torch.ones_like(rows, dtype=torch.float))
    deg = torch.clamp(deg, min=1.0)

    # Normalized adjacency values
    vals = 1.0 / torch.sqrt(deg[rows] * deg[cols])

    # Sparse adjacency matrix
    adj = torch.sparse_coo_tensor(
        indices=torch.stack([rows, cols], dim=0),
        values=vals,
        size=(N, N),
        device=device
    ).coalesce()

    return adj

adj = build_norm_adj(train_edges, n_users, n_items, device=device)
print("Adjacency built.")
print(" - Shape:", tuple(adj.shape))
print(" - Non-zeros (nnz):", adj._nnz())
print(" - Example nnz density:", adj._nnz() / (adj.shape[0] * adj.shape[1]))



PART 6 — Build normalized adjacency (user–item graph)
Using device: cpu
Adjacency built.
 - Shape: (53, 53)
 - Non-zeros (nnz): 156
 - Example nnz density: 0.055535777856888575


In [9]:
# -----------------------------
# PART 7 — Define + train LightGCN (BPR loss)
# -----------------------------
print("\n===================================")
print("PART 7 — Train LightGCN (BPR loss)")
print("===================================")

import numpy as np
import random
import torch

class LightGCN(torch.nn.Module):
    """
    LightGCN for implicit recommendation on a user–item bipartite graph.
    Uses layer-wise embedding propagation + mean pooling across layers.
    """
    def __init__(self, n_users, n_items, emb_dim=64, n_layers=2):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.n_layers = n_layers

        self.user_emb = torch.nn.Embedding(n_users, emb_dim)
        self.item_emb = torch.nn.Embedding(n_items, emb_dim)

        # Small random init
        torch.nn.init.normal_(self.user_emb.weight, std=0.1)
        torch.nn.init.normal_(self.item_emb.weight, std=0.1)

    def forward(self, adj):
        # Concatenate user + item embeddings into one matrix
        all_emb = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)

        # Store embeddings at each layer (including layer 0)
        embs = [all_emb]
        for _ in range(self.n_layers):
            all_emb = torch.sparse.mm(adj, all_emb)
            embs.append(all_emb)

        # Mean pooling over layers
        out = torch.stack(embs, dim=0).mean(dim=0)

        user_out = out[:self.n_users]
        item_out = out[self.n_users:]
        return user_out, item_out


def sample_bpr_batch(train_user_pos, n_users, n_items, batch_size=256):
    """
    Samples (user, positive_item, negative_item) triples for BPR training.
    Negative items are sampled from items the user did NOT positively interact with (in train).
    """
    users = np.random.randint(0, n_users, size=batch_size)
    pos_items = []
    neg_items = []

    for u in users:
        pos_list = list(train_user_pos[u])

        # Safety fallback (should rarely happen)
        if len(pos_list) == 0:
            # pick another random user that has positives
            tries = 0
            while len(pos_list) == 0 and tries < 10:
                u = np.random.randint(0, n_users)
                pos_list = list(train_user_pos[u])
                tries += 1
            if len(pos_list) == 0:
                pos_list = [np.random.randint(0, n_items)]

        p = random.choice(pos_list)

        n = np.random.randint(0, n_items)
        while n in train_user_pos[u]:
            n = np.random.randint(0, n_items)

        pos_items.append(p)
        neg_items.append(n)

    return (
        torch.tensor(users, dtype=torch.long),
        torch.tensor(pos_items, dtype=torch.long),
        torch.tensor(neg_items, dtype=torch.long),
    )


def bpr_loss(u_emb, i_emb, users, pos_items, neg_items, reg=1e-4):
    """
    BPR loss encourages score(u, pos) > score(u, neg).
    Adds small L2 regularization on the batch embeddings.
    """
    u = u_emb[users]
    pos = i_emb[pos_items]
    neg = i_emb[neg_items]

    pos_scores = (u * pos).sum(dim=1)
    neg_scores = (u * neg).sum(dim=1)

    loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()

    # L2 regularization (only on embeddings involved in this batch)
    reg_loss = reg * (u.norm(2).pow(2) + pos.norm(2).pow(2) + neg.norm(2).pow(2)) / users.shape[0]
    return loss + reg_loss


# ---- Hyperparameters (demo-friendly defaults) ----
EMB_DIM = 64
N_LAYERS = 2
LR = 1e-2
REG = 1e-4
EPOCHS = 150
BATCH_SIZE = 256

print("Hyperparameters:")
print(f" - EMB_DIM={EMB_DIM}, N_LAYERS={N_LAYERS}, LR={LR}, REG={REG}")
print(f" - EPOCHS={EPOCHS}, BATCH_SIZE={BATCH_SIZE}")

# ---- Model + optimizer ----
model = LightGCN(n_users, n_items, emb_dim=EMB_DIM, n_layers=N_LAYERS).to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)

# ---- Training loop ----
for epoch in range(1, EPOCHS + 1):
    model.train()

    users, pos_items, neg_items = sample_bpr_batch(
        train_user_pos=train_user_pos,
        n_users=n_users,
        n_items=n_items,
        batch_size=BATCH_SIZE
    )

    users = users.to(device)
    pos_items = pos_items.to(device)
    neg_items = neg_items.to(device)

    u_out, i_out = model(adj)
    loss = bpr_loss(u_out, i_out, users, pos_items, neg_items, reg=REG)

    opt.zero_grad()
    loss.backward()
    opt.step()

    if epoch % 25 == 0 or epoch == 1:
        print(f"Epoch {epoch:>3} | Loss: {loss.item():.4f}")

print("\nTraining complete ✅")



PART 7 — Train LightGCN (BPR loss)
Hyperparameters:
 - EMB_DIM=64, N_LAYERS=2, LR=0.01, REG=0.0001
 - EPOCHS=150, BATCH_SIZE=256
Epoch   1 | Loss: 0.6752
Epoch  25 | Loss: 0.1147
Epoch  50 | Loss: 0.0151
Epoch  75 | Loss: 0.0091
Epoch 100 | Loss: 0.0069
Epoch 125 | Loss: 0.0055
Epoch 150 | Loss: 0.0041

Training complete ✅


In [11]:
# -----------------------------
# PART 8 — Evaluate + show Top-K recommendations 
# -----------------------------
print("\n====================================================")
print("PART 8 — Evaluate + Top-K recommendations")
print("====================================================")

import numpy as np
import torch

# ---- Helper: metrics at K ----
def precision_recall_ndcg_at_k(ranked_items, ground_truth_set, k):
    """
    ranked_items: list of item indices sorted by predicted score (best first)
    ground_truth_set: set of held-out positives (e.g., test items for this user)
    """
    topk = ranked_items[:k]
    hits = [1 if i in ground_truth_set else 0 for i in topk]

    precision = sum(hits) / k
    recall = sum(hits) / max(1, len(ground_truth_set))

    # DCG
    dcg = 0.0
    for idx, h in enumerate(hits, start=1):
        if h:
            dcg += 1.0 / np.log2(idx + 1)

    # IDCG (best possible DCG)
    ideal_len = min(len(ground_truth_set), k)
    if ideal_len == 0:
        ndcg = 0.0
    else:
        idcg = sum([1.0 / np.log2(i + 1) for i in range(2, 2 + ideal_len)])
        ndcg = dcg / idcg

    return precision, recall, ndcg


@torch.no_grad()
def evaluate_at_k(model, adj, train_user_pos, val_user_pos, test_user_pos, k=10):
    """
    Evaluates Precision@K / Recall@K / NDCG@K on test positives.
    Masks train/val items so we don't recommend previously seen positives.
    """
    model.eval()
    u_out, i_out = model(adj)

    precisions, recalls, ndcgs = [], [], []
    evaluated_users = 0

    for u in range(n_users):
        gt = test_user_pos[u]
        if len(gt) == 0:
            continue

        evaluated_users += 1
        scores = (u_out[u] @ i_out.T).detach().cpu().numpy()

        # Mask seen positives (train + val)
        seen = train_user_pos[u].union(val_user_pos[u])
        if len(seen) > 0:
            scores[list(seen)] = -1e9

        ranked = list(np.argsort(-scores))
        p, r, n = precision_recall_ndcg_at_k(ranked, gt, k)
        precisions.append(p)
        recalls.append(r)
        ndcgs.append(n)

    return {
        f"Precision@{k}": float(np.mean(precisions)) if precisions else None,
        f"Recall@{k}": float(np.mean(recalls)) if recalls else None,
        f"NDCG@{k}": float(np.mean(ndcgs)) if ndcgs else None,
        "UsersEvaluated": evaluated_users,
    }


# ---- Run evaluation ----
K = 10
metrics = evaluate_at_k(model, adj, train_user_pos, val_user_pos, test_user_pos, k=K)

print("\nEvaluation metrics:")
for k, v in metrics.items():
    print(f" - {k}: {v}")

# ---- Build joke text lookup for printing recommendations ----
joke_text_lookup = {}
if "joke_text" in df.columns:
    temp = df.dropna(subset=["joke_text"]).drop_duplicates("joke_id")[["joke_id", "joke_text"]]
    joke_text_lookup = dict(zip(temp["joke_id"].astype(str), temp["joke_text"].astype(str)))


@torch.no_grad()
def recommend_topk_for_user(model, adj, user_idx, k=5):
    """
    Returns top-k recommended item indices for a user,
    masking anything they have already rated (train/val/test).
    """
    model.eval()
    u_out, i_out = model(adj)

    scores = (u_out[user_idx] @ i_out.T).detach().cpu().numpy()

    # Mask everything the user has already interacted with (any rating)
    seen_all = set(df[df["user_idx"] == user_idx]["item_idx"].tolist())
    if len(seen_all) > 0:
        scores[list(seen_all)] = -1e9

    ranked = np.argsort(-scores)[:k]
    return ranked


# ---- Print Top-5 recommendations per user ----
print("\nTop-5 recommendations per user:")
for u in range(n_users):
    uid = idx2user[u]
    recs = recommend_topk_for_user(model, adj, u, k=5)

    print("\n----------------------------------------")
    print(f"User: {uid} | Recommended Top-5 jokes")
    print("----------------------------------------")

    for rank, item_idx in enumerate(recs, start=1):
        joke_id = str(idx2item[item_idx])
        text = joke_text_lookup.get(joke_id, "[no joke_text found]")
        print(f"{rank}. joke_id={joke_id} | {text[:140]}{'...' if len(text) > 140 else ''}")

print("\n✅ Part 8 complete: you now have metrics + readable Top-K recommendations")



PART 8 — Evaluate + Top-K recommendations

Evaluation metrics:
 - Precision@10: 0.07500000000000001
 - Recall@10: 0.75
 - NDCG@10: 0.5892640711035394
 - UsersEvaluated: 4

Top-5 recommendations per user:

----------------------------------------
User: A | Recommended Top-5 jokes
----------------------------------------
1. joke_id=103534 | What's the difference between America and yoghurt? If you leave yoghurt alone for 200 years it develops a culture.
2. joke_id=108789 | [DUI checkpoint] Cop: I'm gonna need you to follow my finger Me: As long as it doesn't tweet inspirational stuff
3. joke_id=113773 | Why couldn't Hillary rig the election like she rigged the DNC? She deleted that email.
4. joke_id=117720 | This guy asked me to convert today... And I said, no, I'm happy with my current non-prophet organization
5. joke_id=123726 | whenever i trip a skinny girl running in only a sports bra i feel like i'm doing god's work

----------------------------------------
User: B | Recommended To